Step 1: Install Gradio

In [ ]:
!pip install pandas scikit-learn joblib gradio --pre


Step 2: Import libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
import gradio as gr


Step 3: Load your dataset

In [ ]:
df = pd.read_csv(r"C:\Users\LENOVO\Desktop\Coding\Hospital.csv")
df.head()


,age,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,medical_specialty,diag_1,diag_2,diag_3,glucose_test,A1Ctest,change,diabetes_med,readmitted
0,[70-80),8,72,1,18,2,0,0,Missing,Circulatory,Respiratory,Other,no,no,no,yes,no
1,[70-80),3,34,2,13,0,0,0,Other,Other,Other,Other,no,no,no,yes,no
2,[50-60),5,45,0,18,0,0,0,Missing,Circulatory,Circulatory,Circulatory,no,no,yes,yes,yes
3,[70-80),2,36,0,12,1,0,0,Missing,Circulatory,Other,Diabetes,no,no,yes,yes,yes
4,[60-70),1,42,0,7,0,0,0,InternalMedicine,Other,Circulatory,Respiratory,no,no,no,yes,no


Step 4: Preprocess data

In [ ]:
# Example preprocessing (adjust according to your dataset)
# Encode categorical variables
categorical_cols = ['age', 'glucose_test', 'A1Ctest', 'change', 'diabetes_med']

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # save encoders to decode later if needed

# Features and target
X = df.drop('readmitted', axis=1)
y = df['readmitted']


Step 5: Split data

In [ ]:
# -------------------------------
# Train/Test Split
# -------------------------------

from sklearn.model_selection import train_test_split

# X = features, y = target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check shapes
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (20000, 16)
X_test shape : (5000, 16)
y_train shape: (20000,)
y_test shape : (5000,)


Step 6: Train Random Forest model

In [ ]:
# -------------------------------
# Random Forest Training (Error-Free)
# -------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Example: X = features, y = target
# Identify categorical columns (all string/object columns)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Encode categorical columns
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))  # Convert all to string first
    le_dict[col] = le

# Fill missing values
X.fillna(0, inplace=True)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")


Test Accuracy: 60.50%


Step 7: Create a prediction function

In [ ]:
def predict_readmission(input_dict):
    import pandas as pd
    
    # Convert input to DataFrame
    X_input = pd.DataFrame([input_dict])
    
    # Encode categorical columns using the same LabelEncoders as training
    for col, le in le_dict.items():
        if col in X_input.columns:
            X_input[col] = le.transform(X_input[col].astype(str))
    
    # Fill missing values
    X_input.fillna(0, inplace=True)
    
    # Predict
    prediction = model.predict(X_input)[0]
    
    return "Readmitted" if prediction == 1 else "Not Readmitted"


Step 8: Build Gradio interface

In [ ]:
# -------------------------------
# Step 1: Import Libraries
# -------------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import gradio as gr

# -------------------------------
# Step 2: Load Dataset
# -------------------------------
df = pd.read_csv(r"C:\Users\LENOVO\Desktop\Coding\Hospital.csv")  # Replace with your path

# -------------------------------
# Step 3: Prepare Features and Target
# -------------------------------
X = df.drop("readmitted", axis=1)  # Features
y = df["readmitted"].map({'yes':1, 'no':0})  # Encode target as 0/1

# -------------------------------
# Step 4: Encode Categorical Columns
# -------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
le_dict = {}  # Save encoders

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Fill missing values
X.fillna(0, inplace=True)

# -------------------------------
# Step 5: Train/Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Step 6: Train Random Forest Model
# -------------------------------
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate model
accuracy = model.score(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

# -------------------------------
# Step 7: Prediction Function
# -------------------------------
def predict_readmission(*args):
    feature_names = [
        "age", "time_in_hospital", "n_lab_procedures", "n_procedures",
        "n_medications", "n_outpatient", "n_inpatient", "n_emergency",
        "medical_specialty", "diag_1", "diag_2", "diag_3",
        "glucose_test", "A1Ctest", "change", "diabetes_med"
    ]
    
    X_input = pd.DataFrame([dict(zip(feature_names, args))])
    
    # Encode categorical columns using saved encoders
    for col, le in le_dict.items():
        if col in X_input.columns:
            X_input[col] = le.transform(X_input[col].astype(str))
    
    # Fill missing values
    X_input.fillna(0, inplace=True)
    
    # Predict
    prediction = model.predict(X_input)[0]
    
    return "Readmitted" if prediction == 1 else "Not Readmitted"

# -------------------------------
# Step 8: Gradio Interface with Dropdowns
# -------------------------------

# Get unique values for dropdowns
age_options = df['age'].dropna().unique().tolist()
glucose_options = df['glucose_test'].dropna().unique().tolist()
A1C_options = df['A1Ctest'].dropna().unique().tolist()
change_options = df['change'].dropna().unique().tolist()
diabetes_options = df['diabetes_med'].dropna().unique().tolist()
medical_specialty_options = df['medical_specialty'].dropna().unique().tolist()
diag_options = df['diag_1'].dropna().unique().tolist()  # can reuse for diag_2 & diag_3

inputs = [
    gr.Dropdown(choices=age_options, label="Age Group"),
    gr.Number(label="Time in Hospital"),
    gr.Number(label="Number of Lab Procedures"),
    gr.Number(label="Number of Procedures"),
    gr.Number(label="Number of Medications"),
    gr.Number(label="Number of Outpatient Visits"),
    gr.Number(label="Number of Inpatient Visits"),
    gr.Number(label="Number of Emergency Visits"),
    gr.Dropdown(choices=medical_specialty_options, label="Medical Specialty"),
    gr.Dropdown(choices=diag_options, label="Diagnosis 1"),
    gr.Dropdown(choices=diag_options, label="Diagnosis 2"),
    gr.Dropdown(choices=diag_options, label="Diagnosis 3"),
    gr.Dropdown(choices=glucose_options, label="Glucose Test"),
    gr.Dropdown(choices=A1C_options, label="A1C Test"),
    gr.Dropdown(choices=change_options, label="Change"),
    gr.Dropdown(choices=diabetes_options, label="Diabetes Medication")
]

gr.Interface(
    fn=predict_readmission,
    inputs=inputs,
    outputs=gr.Textbox(label="Readmission Prediction"),
    title="Hospital Readmission Predictor",
    description="Select patient details from dropdowns and input numeric values to predict hospital readmission."
).launch()


Test Accuracy: 60.50%
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
